Analizaremos principios basicos de probabilidad, utilizando el libro de Cien Años de Soledad.
- Espacio de probabilidad, de cada letra
- Espacio de probabilidad, de cada palabra
- Mediante probabilidad condicional, generar una palabra condicionada a las n letras anteriores
- Mediante probabilidad condicional, generar una palabra condicionada a las n palabras anteriores

## Librerias

In [0]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Iterable, Any, Dict, List, Optional
import unicodedata
import os
import re
from random import sample
from operator import itemgetter
from collections import Counter
from __future__ import annotations
from collections import defaultdict
import random
from plotnine import * 
import matplotlib.pyplot as plt

pip install pandas numpy plotnine

## Lectura del libro

In [0]:
def read_book(path):
    with open(path, 'r', encoding='utf8') as f:
        text = f.read()
        text = text.replace('\n', '').replace('\r', '')
    return text

In [0]:
path_book = Path.cwd().parent / 'data' / 'gabriel_garcia_marquez_cien_annos_soledad.txt'

In [0]:
path_book

In [0]:
book = read_book(path_book)
book[:10000]

## Funciones de limpieza del texto

In [0]:
def delete_items(text: str, list_items: Iterable[str]) -> str:
    ''' Remove all occurrences of each substring in `list_items` from `text`.
    This returns a new string where every item found in `list_items` has been removed (replaced with the empty string). 
    The operator is performed sequentially for each item in `list_items`.

    Args:
        text: The source string to process.
        list_items: An iterable of substring to remove from `text`.

    Returns:
        A new string with all occurrences of each item in `list_items` removed.

    Raises:
        TypeError: If `text` is not a string or any element of `list_items` is not a str.

    Examples:
        >>> delete_items('hello world', ['l','o'])
        'he wrd'
        >>> delete_items('abc123abc', ['abc'])
        '123'
    '''
    if not isinstance(text, str):
        raise TypeError('text must be a str')
    items = [i for i in list_items] ## items = list(list_items)
    for item in items:
        if not isinstance(item, str):
            raise TypeError('All items in list must be str')

    items = [re.escape(i) for i in items if i]
    if not items:
        return text
    pattern = '|'.join(items)
    #print(pattern)
    return re.sub(pattern,'',text)

In [0]:
help(delete_items)

In [0]:
delete_items('hola mundo, es lo primero que te enseñan a programar', ['mundo', 'te', 'lo'])

In [0]:
def clean_text(text: str) -> str:
    '''
    Normalize and clean a piece of spanish text.

    The function performs the following steps:

    - Validate that `text` is a string.
    - Convert to lowercase
    - Removes diacritical marks(accents, dieresis) while preserving characters like `ñ`.
    - Replaces any character that is not a lowercase ASCII letter `a-z` or `ñ` with a single space (this removes
     punctuation and digits).
    - Collapses repeated spaces and trim leading/trailing spaces.

    Arg:
        text: Input text to normalize and clean.

    Returns:
        A cleaned string in lowercase with accents removed and non-letter characters replaced by single spaces.

    Examples:
        >>> clean_text('Hola, Mundo! 123')
        'hola mundo'

        >>> clean_text("Canción—niño. ÁÉÍÓÚ ü")
        'cancion nino aeiou u'
    '''
    if not isinstance(text, str):
        raise TypeError('text must be a str')

    text = text.lower() # text in lowercase

    # Remove accents
    normalized = unicodedata.normalize("NFD", text) 
    without_accents = ''.join(
        ch for ch in normalized
        if not unicodedata.combining(ch)
    )

    cleaned = re.sub(r"[^a-zñ]+", " ", without_accents)
    return cleaned.strip()

In [0]:
texto = 'Gabriel García Márquez Cien años de soledad EDITADO POR'
normal= unicodedata.normalize("NFD", texto)

In [0]:
list_to_delete = [
    'Gabriel García Márquez',
    'Cien años de soledad',
    'EDITADO POR "EDICIONES LA CUEVA"',
    'Para J omi García Ascot y María Luisa Elio'
]

In [0]:
book = delete_items(book, list_to_delete)

In [0]:
book[:1000]

In [0]:
clean_book = clean_text(book)

In [0]:
clean_book[:1000]

In [0]:
def text_generate_nwords(
    texto: str,
    n: int = 1,
    tot_letras: int = 100,
    seed: Optional[int] = None
) -> str:
    """Generate text using character n-grams.

    The function builds a simple n-gram model from `texto` where for each
    n-gram (substring of length `n`) we record the list of characters that
    follow it in the source text. Then it generates new text by choosing
    an initial n-gram at random and repeatedly sampling a next character
    from the list of followers for the current n-gram.

    Args:
        texto: Source text used to build the n-gram model. Must contain at
            least `n + 1` characters.
        n: The size of each n-gram (number of characters). Must be >= 1.
        tot_letras: Number of characters to generate (the output length will
            be `n + tot_letras` because the initial n-gram is included).
        seed: Optional integer seed for reproducible random generation.

    Returns:
        A generated string starting with a randomly chosen n-gram from the
        source and followed by `tot_letras` characters sampled from the
        learned follower distributions. If generation halts early because
        an n-gram has no recorded followers, the partially generated text
        is returned.

    Raises:
        TypeError: If `texto` is not a str.
        ValueError: If `n` < 1, `tot_letras` < 0, or `texto` is too short to
                    build any n-grams (needs at least `n + 1` chars).

    Examples:
        >>> genera_texto_nletras("hello world", n=2, tot_letras=10, seed=42)
        'lo worldo w'  # deterministic when seed is provided
    """
    if not isinstance(texto, str):
        raise TypeError("texto must be a str")
    if not isinstance(n, int) or n < 1:
        raise ValueError("n must be an integer >= 1")
    if not isinstance(tot_letras, int) or tot_letras < 0:
        raise ValueError("tot_letras must be a non-negative integer")
    if len(texto) < n + 1:
        raise ValueError("texto must contain at least n + 1 characters to build an n-gram model")

    # reproducible RNG
    rng = random.Random(seed)

    # Build n-gram -> followers map
    followers: Dict[str, List[str]] = defaultdict(list)
    for i in range(len(texto) - n):
        ng = texto[i : i + n]
        followers[ng].append(texto[i + n])

    if not followers:
        # defensive: should not happen because of the earlier length check,
        # but keep a clear error message.
        raise ValueError("no n-grams could be built from the provided text")

    # Start from a random observed n-gram
    ngrama = rng.choice(list(followers.keys()))
    texto_generado = [c for c in ngrama]  # use list for efficient concatenation

    # Generate tot_letras characters (each iteration appends one character)
    for _ in range(tot_letras):
        options = followers.get(ngrama)
        if not options:
            break  # no follower for current ngram -> stop early
        siguiente = rng.choice(options)
        texto_generado.append(siguiente)
        # update current ngram: take last n characters
        ngrama = "".join(texto_generado[-n:])

    return "".join(texto_generado)


In [0]:
text_generate_nwords(clean_book,2)